In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Simulasi tepat dan berbising dengan primitif Qiskit Aer

<details>
<summary><b>Versi pakej</b></summary>

Kod pada halaman ini dibangunkan menggunakan keperluan berikut.
Kami mengesyorkan penggunaan versi ini atau yang lebih baharu.

```
qiskit[all]~=2.3.0
qiskit-aer~=0.17
```
</details>
[Simulasi tepat dengan primitif Qiskit](/guides/simulate-with-qiskit-sdk-primitives) menunjukkan cara menggunakan primitif rujukan yang disertakan dengan Qiskit untuk melakukan simulasi tepat bagi Circuit kuantum. Pemproses kuantum sedia ada mengalami ralat, atau hingar, jadi keputusan simulasi tepat tidak semestinya mencerminkan keputusan yang dijangkakan apabila menjalankan Circuit pada perkakasan sebenar. Walaupun primitif rujukan dalam Qiskit tidak menyokong pemodelan hingar, [Qiskit Aer](https://qiskit.org/ecosystem/aer/) menyertakan pelaksanaan primitif yang menyokong pemodelan hingar. Qiskit Aer adalah simulator Circuit kuantum berprestasi tinggi yang boleh anda gunakan sebagai pengganti primitif rujukan untuk prestasi yang lebih baik dan lebih banyak ciri. Ia adalah sebahagian daripada [Ekosistem Qiskit](https://qiskit.github.io/ecosystem/). Dalam artikel ini, kami menunjukkan penggunaan primitif Qiskit Aer untuk simulasi tepat dan berbising.

> **Note:** - `qiskit-aer` v0.14 atau lebih baharu diperlukan.
> - Walaupun primitif Qiskit Aer melaksanakan antara muka primitif, ia tidak menyediakan pilihan yang sama seperti primitif Qiskit Runtime. Tahap ketahanan, sebagai contoh, tidak tersedia dengan primitif Qiskit Aer.
> - Lihat [dokumentasi AerSimulator](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator) untuk butiran tentang pilihan kaedah simulasi yang disokong Aer.

Untuk meneroka simulasi tepat dan berbising, cipta Circuit contoh pada lapan Qubit:

In [1]:
from qiskit.circuit.library import efficient_su2

n_qubits = 8
circuit = efficient_su2(n_qubits)
circuit.draw("mpl")

<Image src="../docs/images/guides/simulate-with-qiskit-aer/extracted-outputs/df70b5fd-971d-4e7d-a23a-8df037c0fa47-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/simulate-with-qiskit-aer/extracted-outputs/df70b5fd-971d-4e7d-a23a-8df037c0fa47-0.svg)

Circuit ini mengandungi parameter untuk mewakili sudut putaran bagi Gate $R_y$ dan $R_z$. Apabila mensimulasikan Circuit ini, kita perlu menentukan nilai eksplisit untuk parameter-parameter ini. Dalam sel seterusnya, kami menentukan beberapa nilai untuk parameter ini dan menggunakan primitif Estimator daripada Qiskit Aer untuk mengira nilai jangkaan tepat bagi boleh cerapan $ZZ \cdots Z$.

In [2]:
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import EstimatorV2 as Estimator

observable = SparsePauliOp("Z" * n_qubits)
params = [0.1] * circuit.num_parameters

exact_estimator = Estimator()
# The circuit needs to be transpiled to the AerSimulator target
pass_manager = generate_preset_pass_manager(3, AerSimulator())
isa_circuit = pass_manager.run(circuit)
pub = (isa_circuit, observable, params)
job = exact_estimator.run([pub])
result = job.result()
pub_result = result[0]
exact_value = float(pub_result.data.evs)
exact_value

0.8870140234256602

Now, let's initialize a noise model that includes depolarizing error of 2% on every CX gate. In practice, the error arising from the two-qubit gates, which are CX gates here, are the dominant source of error when running a circuit. See [Build noise models](./build-noise-models) for an overview of constructing noise models in Qiskit Aer.

In the next cell, we construct an Estimator that incorporates this noise model and use it to compute the expectation value of the observable.

In [3]:
from qiskit_aer.noise import NoiseModel, depolarizing_error

noise_model = NoiseModel()
cx_depolarizing_prob = 0.02
noise_model.add_all_qubit_quantum_error(
    depolarizing_error(cx_depolarizing_prob, 2), ["cx"]
)

noisy_estimator = Estimator(
    options=dict(backend_options=dict(noise_model=noise_model))
)
job = noisy_estimator.run([pub])
result = job.result()
pub_result = result[0]
noisy_value = float(pub_result.data.evs)
noisy_value

0.7247404214143528

Sekarang, mari inisialisasi model hingar yang menyertakan ralat depolarizing sebanyak 2% pada setiap Gate CX. Dalam praktiknya, ralat yang timbul daripada Gate dua Qubit, yang merupakan Gate CX di sini, adalah sumber ralat dominan apabila menjalankan Circuit. Lihat [Bina model hingar](./build-noise-models) untuk gambaran keseluruhan tentang membina model hingar dalam Qiskit Aer.

Dalam sel seterusnya, kami membina Estimator yang menggabungkan model hingar ini dan menggunakannya untuk mengira nilai jangkaan bagi boleh cerapan.

In [4]:
cx_count = circuit.count_ops()["cx"]
(1 - cx_depolarizing_prob) ** cx_count

0.6542558123199923

This value, 65%, gives a rough estimate of the probability that our final state is correct. It is a conservative estimate because it does not take into account the initial state of the simulation.

The following code cell shows how to use the Sampler primitive from Qiskit Aer to sample from the noisy circuit. We need to add measurements to the circuit before running it with the Sampler primitive.

In [5]:
from qiskit_aer.primitives import SamplerV2 as Sampler

measured_circuit = circuit.copy()
measured_circuit.measure_all()

noisy_sampler = Sampler(
    options=dict(backend_options=dict(noise_model=noise_model))
)
# The circuit needs to be transpiled to the AerSimulator target
pass_manager = generate_preset_pass_manager(3, AerSimulator())
isa_circuit = pass_manager.run(measured_circuit)
pub = (isa_circuit, params, 100)
job = noisy_sampler.run([pub])
result = job.result()
pub_result = result[0]
pub_result.data.meas.get_counts()

{'00100000': 1,
 '00000000': 65,
 '10101000': 1,
 '10000000': 5,
 '00001000': 1,
 '00000110': 2,
 '11110010': 1,
 '00000011': 3,
 '01010000': 3,
 '11000000': 3,
 '01111000': 1,
 '01000000': 2,
 '00000010': 1,
 '01100000': 1,
 '00011000': 1,
 '00111100': 1,
 '00010100': 1,
 '00001111': 1,
 '00110000': 1,
 '01100101': 1,
 '00000100': 1,
 '10100000': 1,
 '00000001': 1,
 '11010000': 1}

Seperti yang dapat dilihat, nilai jangkaan dengan kehadiran hingar adalah jauh daripada nilai yang betul. Dalam praktiknya, anda boleh menggunakan pelbagai teknik pengurangan ralat untuk mengatasi kesan hingar, tetapi perbincangan tentang teknik-teknik ini adalah di luar skop artikel ini.

Untuk mendapat gambaran kasar tentang bagaimana hingar mempengaruhi keputusan akhir, pertimbangkan model hingar kita, yang menambahkan ralat depolarizing sebanyak 2% kepada setiap Gate CX. Ralat depolarizing dengan kebarangkalian $p$ ditakrifkan sebagai saluran kuantum $E$ yang mempunyai tindakan berikut pada matriks ketumpatan $\rho$:

$$
E(\rho) = (1 - p) \rho + p\frac{I}{2^n}
$$

di mana $n$ adalah bilangan Qubit, dalam kes ini, 2. Iaitu, dengan kebarangkalian $p$, keadaan digantikan dengan keadaan campuran sepenuhnya, dan keadaan dipelihara dengan kebarangkalian $1 - p$. Selepas $m$ penggunaan saluran depolarizing, kebarangkalian keadaan dipelihara adalah $(1 - p)^m$. Oleh itu, kita jangkakan kebarangkalian mengekalkan keadaan yang betul pada akhir simulasi akan menurun secara eksponen dengan bilangan Gate CX dalam Circuit kita.

Mari hitung bilangan Gate CX dalam Circuit kita dan kira $(1 - p)^m$. Kita panggil `count_ops` untuk mendapatkan kamus yang memetakan nama Gate kepada kiraan, dan dapatkan semula entri untuk Gate CX.